1. Write a robust file reader function:
- Use pathlib.Path to handle file paths.
- The function should be able to handle both UTF-8 and ISO-8859-1/Latin-1 encoding. You may assume that these are the only two     encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
- Return a tuple containing two strings: (file_text_content,encoding_used) .
2. Write a log parser function that uses regular expressions. Log files contain valid saleslines mixed with system noise. Valid line format:
- [2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
- Noise lines to ignore:
    --- PRINTER PAPER LOW WARNING ---
    === NETWORK SYNC OK ===
- Construct a regular expression ( re module) to capture the 6 key fields from
    - valid lines:
        timestamp (string, format: YYYY-MM-DD HH:MM:SS )
        register_id (string, e.g., POS_01 )
        item (string, e.g., Latte )
        price (float, e.g., 4.50 )
        currency (string, e.g., EUR , NOK , USD )
        status (string: SUCCESS , REFUND , or CANCELLED )
- Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

In [364]:
from pathlib import Path
import re


def file_reader(file_path: Path) -> tuple:
    if not Path(file_path).exists():
        raise Exception(f"File Does not exist {file_path}")
    raw = Path(file_path).read_bytes()
    try:
        raw.decode("utf-8")
        enc = "utf-8"
    except UnicodeDecodeError:
        enc = "iso-8859-1"
    with open(file_path, mode="r", encoding=enc) as f:
        data = f.read()
    return (data, enc)


def log_parser(data: list) -> list[dict]:
    # split the text -> separate lines
    valid_lines = []
    lines = data.split("\n")
    # print(len(lines))
    # remove noise lines
    for line in lines:
        # print(line)
        if re.match(r"\[\d+\-\d+\-\d+ \d+\:\d+\:\d+\]", line) is not None:
            # print(line)
            # pick out six values per line
            timestamp = re.search(r"\d+\-\d+\-\d+\s\d+\:\d+\:\d+", line).group(0)
            register_id = re.search(r"\w+\_\d+", line).group(0)
            item = re.search(r"(item=)(\w+)", line).group(2)
            price = re.search(r"\d+\.\d+", line).group(0)
            currency = re.search(r"(\s)(\w{3})(\s)", line).group(2)
            status = re.search(r"(\()(\w+)(\))", line).group(2)
            times = {"timestamp": timestamp,
                     "register_id": register_id,
                     "item": item,
                     "price": price,
                     "currency": currency,
                     "status": status
                     }
            valid_lines.append(times)
    # return all lines without noise
    return valid_lines


values = file_reader("Data/2026/library_cafe/POS_02/register_sales.log")
# print(values)
parsed_line = log_parser(values[0])
print(parsed_line)
print(type(parsed_line))

[{'timestamp': '2026-10-15 07:01:20', 'register_id': 'POS_02', 'item': 'Latte', 'price': '4.02', 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:01:35', 'register_id': 'POS_02', 'item': 'Espresso', 'price': '2.87', 'currency': 'EUR', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:02:32', 'register_id': 'POS_02', 'item': 'Cappuccino', 'price': '3.79', 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:04:51', 'register_id': 'POS_02', 'item': 'Croissant', 'price': '4.17', 'currency': 'EUR', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:06:44', 'register_id': 'POS_02', 'item': 'Espresso', 'price': '3.19', 'currency': 'NOK', 'status': 'CANCELLED'}, {'timestamp': '2026-10-15 07:07:57', 'register_id': 'POS_02', 'item': 'Muffin', 'price': '3.25', 'currency': 'USD', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:08:54', 'register_id': 'POS_02', 'item': 'Espresso', 'price': '2.99', 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-1

Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of pathlib.Path objects:
- Convert the input path to a pathlib.Path object.
- Iterate through items using Path.iterdir() .
- If an item is a directory, recursively search inside it.
- If an item is a file with a matching extension ( .log ), add its Path object to a list.
- Constraint: Do NOT use Path.glob() , Path.rglob() , or os.walk() . You must implement explicit recursion.

2. Metadata about the different stores can be found in the data/store_config.json file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
- Traverses any combination of nested dictionaries and lists found in this file.
- Returns a set (type hint: set[str] ) of all register id strings where "active":
  true .

Example config.json:
{
"company": "Campus Coffee",
"regions": [
{
"region_name": "Main Campus",
"stores": [
{
"name": "Library Cafe",
"registers": [
{"id": "POS_01", "active": true},
{"id": "POS_02", "active": false}
]
}
]
}
]


In [365]:
from pathlib import Path
import json

def traverse_file_system(current_path: Path, path_list: list, file_suffix=".log") -> list:
    p = Path(current_path)
    for dir in p.iterdir():
        if dir.is_dir():
            traverse_file_system(dir, path_list, file_suffix)
        else:
            if dir.suffix == file_suffix:
                # print(dir)
                path_list.append(str(dir))
    return path_list


log_files = traverse_file_system("Data", [])
print(log_files)

"""
def traverse_json(node, path=""):
    if isinstance(node, dict):
        for k, v in node.items():
            if k == "id":
                print("yay!")
            traverse_json(v, f"{path}/{k}")
    elif isinstance(node, list):
        for i, v in enumerate(node):
            traverse_json(v, f"{path}[{i}]")
    else:
        print(path, "=", node)
"""


def find_active_registers(node) -> dict:
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "registers" and isinstance(value, list):
                for i, reg in enumerate(value):
                    if reg["active"]:
                        yield (reg)
            else:
                yield from find_active_registers(value)
    elif isinstance(node, list):
        for i, value in enumerate(node):
            yield from find_active_registers(value)


config = json.loads(Path("Data/2026/store_config.json").read_text(encoding="utf-8"))

for value in find_active_registers(config):
    print(value)


['Data/2026/science_building/POS_01/register_sales.log', 'Data/2026/science_building/POS_03/register_sales.log', 'Data/2026/science_building/POS_02/register_sales.log', 'Data/2026/student_union/POS_01/register_sales.log', 'Data/2026/student_union/POS_03/register_sales.log', 'Data/2026/student_union/POS_02/register_sales.log', 'Data/2026/library_cafe/POS_01/register_sales.log', 'Data/2026/library_cafe/POS_03/register_sales.log', 'Data/2026/library_cafe/POS_02/register_sales.log']
{'id': 'POS_01', 'active': True}
{'id': 'POS_02', 'active': True}
{'id': 'POS_03', 'active': True}


Task 3: Functions for cleaning and analysing data
using NumPy
1. Write a function for removing price outliers that:
- Converts input price data to a 1D NumPy float array.
- Detects hardware glitch outliers ( ) where a value deviates from the mean by more than 3 standard deviations:
    |x-µ| > 3*sigma
- Replaces detected outlier values with the median of the valid prices.
- Returns the cleaned 1D array.

2. Write a function compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray that:
- Computes the element-wise refund ratio using NumPy array operations:
    refund ratio = refund count/(sale count + 10**-6)
- The addition of 10**-6 prevents zero-division errors

In [366]:
import numpy as np


def remove_outliers(price_lines: list[dict]) -> np.ndarray:
    item_prices = []
    for i in price_lines:
        item_price = float(i["price"])
        item_prices.append(item_price)
    item_prices_array = np.array(item_prices)

    sd = np.std(item_prices_array)
    mean_price = np.mean(item_prices_array)
    prices_no_outliers = []
    valid = [i for i in item_prices if np.abs(i - mean_price) <= 3 * sd]
    median_price = np.median(valid)

    for p in item_prices:
        if np.abs(p - mean_price) > 3 * sd:
            prices_no_outliers.append(median_price)
        else:
            prices_no_outliers.append(p)
    median_price = np.median(prices_no_outliers)
    final_prices = np.array(prices_no_outliers)

    return final_prices


remove_outliers(parsed_line)


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    refund_ratio = refund_count / (sales_count + 10**-6)
    # print(refund_ratio)
    return refund_ratio


Task 4: Data analysis using Pandas
1. Write a function that:
- Converts the list of transaction dictionaries read from the log files into a pandas.DataFrame.
- Converts the timestamp column to datetime and sets it as the DataFrame index.
- Sorts the DataFrame chronologically by timestamp.

2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).

3. Write a function that has the DataFrame as input and creates a pivot table displaying total transaction revenue grouped by item across status categories (SUCCESS, REFUND, CANCELLED).

4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

In [367]:
import pandas as pd


def log_to_dataframe(log: list[dict]) -> pd.DataFrame:
    transaction_time = [pd.to_datetime(p["timestamp"]) for p in log]
    transaction_price = [float(p["price"]) for p in log]
    transaction_item = [p["item"] for p in log]
    transaction_status = [p["status"] for p in log]
    transaction_currency = [p["currency"] for p in log]
    transaction_df = pd.DataFrame({"time": transaction_time,
                  "price": transaction_price,
                  "item": transaction_item,
                  "status": transaction_status,
                  "currency": transaction_currency})
    transaction_df = transaction_df.set_index("time")
    transaction_df = transaction_df.sort_values("time")
    # print(transaction_df)
    return transaction_df


transaction_info = log_to_dataframe(parsed_line)
print(transaction_info)

def sum_total_per_item_per_hour(dataframe: pd.DataFrame) -> pd.DataFrame:
    grouped_items = dataframe.groupby(["item", pd.Grouper(freq="1h")])[["price"]].sum()
    # pivot_grouped_items = grouped_items.pivot_table(index="time", columns=["price", "item"])
    # print(pivot_grouped_items)
    return grouped_items

sum_total_per_item_per_hour(transaction_info)

def pivot_by_transaction_status(dataframe: pd.DataFrame) -> pd.DataFrame:
    pivot_table = dataframe.pivot_table(index="item", columns="status", values="price", aggfunc="sum")
    print(pivot_table)
    return pivot_table

pivot_by_transaction_status(transaction_info)


"""def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np:
    refund_ratio = np.size(refund_count) / (np.size(sales_count) + 10**-6)
    return refund_ratio
"""


def refund_ratio_per_item(dataframe: pd.DataFrame) -> list[dict]:
    counts = dataframe.pivot_table(index="item", columns="status", values="price", aggfunc="count", fill_value=0)
    success_counts = counts["SUCCESS"].to_numpy()
    refund_counts = counts["REFUND"].to_numpy()

    print(success_counts)
    print(refund_counts)

    refund_ratio = compute_refund_ratio(success_counts, refund_counts)
    print(refund_ratio)

    refund_ratio_list = []
    for i, item in enumerate(counts.index):
        print(item)
        refund_ratio_list.append({"item": item,
                                  "refund_ratio": refund_ratio[i]})
    print(refund_ratio_list)

    return refund_ratio_list


# refund_ratio_per_item(transaction_info)

                      price        item     status currency
time                                                       
2026-10-15 07:01:20    4.02       Latte    SUCCESS      USD
2026-10-15 07:01:35    2.87    Espresso    SUCCESS      EUR
2026-10-15 07:02:32    3.79  Cappuccino    SUCCESS      NOK
2026-10-15 07:04:51    4.17   Croissant     REFUND      EUR
2026-10-15 07:06:44    3.19    Espresso  CANCELLED      NOK
2026-10-15 07:07:57    3.25      Muffin     REFUND      USD
2026-10-15 07:08:54    2.99    Espresso    SUCCESS      NOK
2026-10-15 07:10:09  999.00       Latte    SUCCESS      NOK
2026-10-15 07:11:41    4.26   Croissant    SUCCESS      NOK
2026-10-15 07:13:41    3.10    Espresso    SUCCESS      EUR
2026-10-15 07:14:22  999.00    Espresso  CANCELLED      NOK
2026-10-15 07:14:34    7.82    Sandwich     REFUND      EUR
2026-10-15 07:16:56    4.19   Croissant     REFUND      NOK
2026-10-15 07:19:48    3.20      Muffin     REFUND      NOK
2026-10-15 07:21:25    7.37    Sandwich 

Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove price glitches.
5. Apply currency correction: multiply price by 10 for all transactions where currency == "NOK".
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and
outputs for this assignment).

Outputs (files your code should generate)
- cleaned_transactions.csv : A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
- hourly_revenue.json : A JSON report containing total revenue per hour per item.
- status_pivot_table.csv : A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
- item_refund_ratios.csv : A CSV table containing refund ratios for each itemtype.

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


all_json_files = traverse_file_system("Data", [], ".json")
# print(all_json_files)

all_active_registers = []
for json_file in all_json_files:
    json_config = json.loads(Path(json_file).read_text(encoding="utf-8"))
    active_registers = [register["id"] for register in find_active_registers(json_config)]
    all_active_registers.extend(active_registers)
# print(all_active_registers)

all_log_files = traverse_file_system("Data", [], ".log")
# print(all_log_files)

all_sales_information = []
for log_file in all_log_files:
    values = file_reader(log_file)
    sales_information = log_parser(values[0])
    all_sales_information.extend(sales_information)
# print(all_sales_information)


cleaned_prices = remove_outliers(all_sales_information)
for record, price in zip(all_sales_information, cleaned_prices):
    record["price"] = price

dataframe = log_to_dataframe(all_sales_information)
dataframe.loc[dataframe["currency"] == "NOK", "price"] *= 10

dataframe.to_csv("cleaned_transactions.csv", mode="w")

# with open("cleaned_transactions.csv", mode="w+") as f:
#     print(dataframe, file=f)


resampled = sum_total_per_item_per_hour(dataframe)
print(resampled)
resampled["price"].unstack("item").to_json("hourly_revenue.json", date_format="iso", indent=4, mode="w")

pivoted_data = pivot_by_transaction_status(dataframe)
print(pivoted_data)
pivoted_data.to_csv("status_pivot_table.csv", mode="w")

refund_ratio_data = refund_ratio_per_item(dataframe)
print(refund_ratio_data)
with open("item_refund_ratios.csv", mode="w+") as f:
    print("item, refund_ratio", file=f)
    for e in refund_ratio_data:
        print(f"{e['item']}, {e['refund_ratio']:.2f}", file=f)


                                  price
item       time                        
Cappuccino 2026-10-15 07:00:00   916.12
           2026-10-15 08:00:00  1061.36
           2026-10-15 09:00:00    89.18
Croissant  2026-10-15 07:00:00   932.17
           2026-10-15 08:00:00  1077.01
           2026-10-15 09:00:00    22.46
Espresso   2026-10-15 07:00:00  1014.08
           2026-10-15 08:00:00   947.96
           2026-10-15 09:00:00    89.32
Latte      2026-10-15 07:00:00   810.27
           2026-10-15 08:00:00  1401.61
           2026-10-15 09:00:00     7.78
Muffin     2026-10-15 07:00:00  1231.07
           2026-10-15 08:00:00   773.28
           2026-10-15 09:00:00     3.18
Sandwich   2026-10-15 07:00:00  1200.13
           2026-10-15 08:00:00  1078.74
           2026-10-15 09:00:00    86.96
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino     336.29  469.17  1261.20
Croissant      280.24  333.27  1418.13
Espresso       340.35  457.58  1253.43
Latte